# Evidence, Not Doubt: Self-Verification in LLM-Based Arabic Entity Linking to Wikidata

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahammediAnesAbdelouahab/arabic-el-agent/blob/main/notebooks/reproduce_and_run.ipynb)

An LLM agent that links Arabic entity mentions to Wikidata and **checks its answers, mostly against Wikidata,** before accepting them, together with the comparison systems, run traces and review tools of the paper.

**Findings of the paper**
1. **Evidence, not doubt.** In the same correction loop, checks computed from Wikidata repaired answers and rarely broke any, whereas a general LLM critic (`critic_only`) broke many correct answers.
2. **A rule instead of a loop.** Nearly all repairs came from the NIL check, and a fixed rule acting on its trigger (`llm_plus_nil_rule`) did as well without extra model calls.
3. **Labels matter more.** ELNER-DZ leaves dates unlinked, and a blind pooled review found other wrong labels in our test sample; both effects are larger than the gain from verification.

**Pipeline for every mention**
1. **Candidate retrieval:** Wikidata search, Arabic Wikipedia titles, redirects and disambiguation pages, clitic-aware query variants (e.g. lil-Jaza'ir → al-Jaza'ir), and English/French search for mentions written in Latin script.
2. **Decision:** the LLM picks a QID from the candidate list, answers NIL, or asks for a new search.
3. **Verification:** the checks below.
4. **Correction:** any problem is sent back to the LLM together with its evidence, for at most three rounds.

| Check | What it verifies | Hard? |
|---|---|---|
| `FORMAT` | the answer is valid JSON with an action and, for a selection, a QID | yes |
| `CANDIDATE` | the QID comes from the candidate list (no invented IDs) | yes |
| `NOT_ENTITY` | the item is not a disambiguation page, category, template or list | yes |
| `TYPE` | the NER type is compatible with P31/P279* (SPARQL) | no (metonymy can be confirmed) |
| `LABEL` | the mention matches an Arabic/English/French label or alias (low confidence only) | no |
| `NIL` | NIL is challenged when an exact, type-compatible candidate exists | no |
| `COHERENCE` | the links of one text are consistent with each other (judged by the LLM) | no |
| `ROLE` (variant) | an LLM critic for metonymy (country → national team) | no |
| `CRITIC` (variant) | a general LLM critic that judges any answer from the text and its own knowledge | no |

**This notebook**
- **Part A:** reproduces every table of the paper from the released run files. No API key is needed.
- **Part B:** runs the agent on the challenge set and on the corrected ELNER-DZ test set. An LLM API key is needed.
- **Part C:** uses the agent on your own data.
- **Part D:** reviews disagreements with the pooled, blind review tools used for the paper.

## Setup

In [ ]:
# Clone the repository (on Colab) and install the dependencies.
import os, sys
REPO_URL = "https://github.com/MahammediAnesAbdelouahab/arabic-el-agent.git"
REPO_DIR = "/content/arabic-el-agent" if os.path.isdir("/content") else os.path.abspath("..")
if not os.path.exists(os.path.join(REPO_DIR, "elagent.py")):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))
from elagent import *
import pandas as pd
print("elagent", ELAGENT_VERSION, "|", REPO_DIR)

## Part A: Reproduce the paper's tables (no API key)

Everything is recomputed from `results/runs/` (each system's answers plus the full agent traces) and `results/review/` (the manual review).

**Tables of the paper:**
- `table2_first_runs_original` / `table2_first_runs_corrected`: the first run of every system against the original and the corrected labels (Table 2).
- `table3_within_run_corrected` / `table3_within_run_no_dates`: repairs and breaks inside each verification run, on all scored mentions and without the date mentions, which ELNER-DZ does not link (Table 3). Repeated runs carry the tag `@r2`; runs with another model carry the model name.
- `table3_llm_only_spread`: the first answer of every main-model run is an independent LLM-only run; their spread measures run-to-run variation.
- `table4_checks_<run>`: which checks fired and what happened afterwards (Table 4).
- `table5_label_review`: labels the review could not accept, with the dates apart (Table 5).
- `table6_popularity`: first runs by popularity bucket (Table 6).
- `nil_check_vs_rule`, `by_script`, `gold_vs_agent_vs_search`: further analyses quoted in the text.

**Columns:**
- `fixed` / `broken`: answers changed within a run, wrong → right and right → wrong; `sign_test_p` is the exact sign test.
- `wins_vs_ref` / `losses_vs_ref`, `p_vs_ref` (exact McNemar test) and `diff_vs_ref_95CI` (bootstrap): comparison with `llm_no_correction` on the same mentions.

In [ ]:
import reproduce_tables as R
tables = R.reproduce(os.path.join(REPO_DIR, "results", "runs", "elner"),
                     os.path.join(REPO_DIR, "results", "review", "review_pooled.csv"),
                     out_dir=None, verbose=False)
for name in ["table2_first_runs_original", "table2_first_runs_corrected", "table3_within_run_corrected",
             "table3_within_run_no_dates", "table3_llm_only_spread", "table4_checks_full_no_memory",
             "table4_checks_critic_only", "table5_label_review", "table6_popularity", "nil_check_vs_rule"]:
    if name in tables:
        print(f"\n=== {name} ===")
        display(tables[name])

## Part B: Run the agent (needs an LLM API key)

**API key:** in Colab, open the key icon (Secrets) and add `GEMINI_API_KEY`, or `ANTHROPIC_API_KEY` / `OPENAI_API_KEY`. Then enable notebook access.

**Settings:**
- **Model:** the paper used `gemini-3.8-flash` with thinking level `low`. The second model, Qwen3-4B-Instruct-2507 (4-bit), was served locally with Ollama through its OpenAI-compatible endpoint: set `llm_backend="openai"`, `llm_base_url="http://localhost:11434/v1"` and pass `system_suffix=" /no_think"` to `LLM`.
- **Wikidata snapshot:** the released cache `cache/wikidata_cache.json.gz` was collected on 2026-10-03 to 2026-10-05. It is used here so that candidate retrieval sees the same Wikidata as in the paper. Requests for anything not in the cache go to the live Wikidata.
- **Contact e-mail:** put yours in `user_agent`, as the Wikimedia API policy asks.

**Expected differences:** LLM outputs are not deterministic, so a new run will differ slightly from the released runs. The released runs are the reference for the paper's numbers.

In [ ]:
import shutil
WORK_DIR = "/content/elagent_work" if os.path.isdir("/content") else os.path.abspath("elagent_work")
os.makedirs(os.path.join(WORK_DIR, "cache"), exist_ok=True)
CACHE = os.path.join(WORK_DIR, "cache", "wikidata_cache.json.gz")
if not os.path.exists(CACHE) and os.path.exists(os.path.join(REPO_DIR, "cache", "wikidata_cache.json.gz")):
    shutil.copy(os.path.join(REPO_DIR, "cache", "wikidata_cache.json.gz"), CACHE)

CFG = Config(
    project_dir=WORK_DIR,
    llm_backend="gemini",              # "gemini" | "anthropic" | "openai" (any OpenAI-compatible server)
    llm_model="gemini-3.8-flash",
    gemini_thinking_level="low",
    min_llm_interval=0.0,              # raise it (e.g. 4.0) if you hit rate limits
    user_agent="ArabicELAgent/1.0 (research; contact: your_email@example.com)",   # <- your e-mail
)
KEY_NAME = {"gemini": "GEMINI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}[CFG.llm_backend]
try:
    from google.colab import userdata
    API_KEY = userdata.get(KEY_NAME)
except Exception:
    API_KEY = os.environ.get(KEY_NAME)
assert API_KEY, f"Add {KEY_NAME} to Colab Secrets (key icon) or to the environment."

RUNS = os.path.join(WORK_DIR, "runs")
wd = WikidataClient(CFG.user_agent, cache_path=CACHE)
llm = LLM(CFG.llm_backend, CFG.llm_model, api_key=API_KEY, base_url=CFG.llm_base_url, temperature=CFG.temperature,
          thinking_level=CFG.gemini_thinking_level if CFG.llm_backend == "gemini" else None,
          min_interval=CFG.min_llm_interval)
agent = Agent(replace(CFG, use_memory=False), wd, llm)
print("ready:", CFG.llm_backend, CFG.llm_model)

### Quick demo: the trace shows the candidates, every check and any correction

In [ ]:
# Two example inputs: a company in a business sentence, and two clubs in a match report.
show(agent.link(Mention(text="أعلنت أمازون عن افتتاح مركز بيانات جديد في الشرق الأوسط.", mention="أمازون", type="ORG")))
print()
t = "فاز برشلونة على ريال مدريد في مباراة الكلاسيكو."
for r in agent.link_document([Mention(t, "برشلونة", "ORG"), Mention(t, "ريال مدريد", "ORG")]):
    show(r); print()

### Challenge set (59 hand-verified hard mentions)

`data/challenge_set.jsonl` covers these categories:
- ambiguous names
- partial names
- nicknames
- metonymy
- dialect/Arabizi
- transliterations
- historical nisbas
- NIL namesakes

In [ ]:
CHALLENGE = load_rows(os.path.join(REPO_DIR, "data", "challenge_set.jsonl"))
runs = {}
for name, cfg in [("challenge_no_correction", replace(CFG, use_memory=False, max_rounds=0)),
                  ("challenge_full", replace(CFG, use_memory=False))]:
    _, runs[name] = evaluate(Agent(cfg, wd, llm), CHALLENGE, name, RUNS)
cols = ["n", "accuracy", "fixed (wrong→right)", "broken (right→wrong)", "candidate_recall"]
pd.concat({n: metrics_by(r)[cols] for n, r in runs.items()}, axis=1)

### Corrected ELNER-DZ test set

`data/elner_dz_test_corrected.jsonl` gives two labels for every mention:
- `gold_original`: the ELNER-DZ label.
- `gold_corrected`: the list of acceptable QIDs after the review.

`label_status` explains each label (see `data/README.md`). Mentions with status `unclear` are excluded from scoring.

If the file was released without sentence texts, download ELNER-DZ yourself and set `ELNER_FILE`. The texts are then restored by sentence id and checked by SHA-1.

In [ ]:
TEST = load_rows(os.path.join(REPO_DIR, "data", "elner_dz_test_corrected.jsonl"))
ELNER_FILE = None                       # path to the original ELNER-DZ JSON, only if the texts are missing
if "text" not in TEST[0]:
    assert ELNER_FILE, "This release has no texts: set ELNER_FILE to the original ELNER-DZ file."
    restore_texts(TEST, ELNER_FILE)
rows = [{**r, "gold": r["gold_original"], "category": r.get("popularity")} for r in TEST]
acceptable, unclear = corrected_labels(TEST)

LIMIT = 100                             # start small; None = all mentions
CONFIGS = ["baseline_search_top1", "llm_no_correction", "full_no_memory", "grounded_only", "critic_only"]
run_ablations(rows, CFG, wd, llm, names=CONFIGS, out_dir=os.path.join(RUNS, "elner"), limit=LIMIT)
my_runs = {n: os.path.join(RUNS, "elner", n + ".jsonl") for n in CONFIGS}

# The fixed NIL rule is built from the LLM run without any model call.
save_rows(apply_nil_rule(load_rows(my_runs["llm_no_correction"]), wd), os.path.join(RUNS, "elner", "llm_plus_nil_rule.jsonl"))
my_runs["llm_plus_nil_rule"] = os.path.join(RUNS, "elner", "llm_plus_nil_rule.jsonl")
paired_table(my_runs, ref="llm_no_correction", acceptable=acceptable, exclude=unclear)

## Part C: Your own data

**Reading a dataset:** `iter_records` reads JSON, JSON-lines, CSV or CoNLL files, even if they are gzip-compressed or very large. `detect_schema` then guesses the field names, and `convert_records` turns the records into one row per mention. Check the detected schema before converting:

In [ ]:
# records = list(iter_records("my_dataset.json", limit=300))
# schema = detect_schema(records)            # inspect, edit if a field was guessed wrong
# rows = convert_records(iter_records("my_dataset.json"), schema)
# add_popularity(rows, wd)                   # head / torso / tail buckets by Wikidata sitelinks
# dev_rows, test_rows = split_rows(rows, dev_frac=0.2, seed=13)

# A silver set built from human-made links in random Arabic Wikipedia articles:
# wiki_rows = build_wiki_link_set(wd, n_mentions=600)

## Part D: Review disagreements (pooled and blind)

These are the tools used for the paper:
- `export_pooled_review` collects every mention where **any** system disagrees with the gold label, plus a random audit sample of agreements.
- `review_blind` shows the distinct answers in a random order with their sources hidden. The reviewer ticks every correct one.
- `rescore_pooled` scores all runs against the corrected labels.
- `agreement_sample` and `agreement` compute Cohen's kappa against a second annotator.

In [ ]:
# REVIEW = os.path.join(WORK_DIR, "review_pooled.csv")
# export_pooled_review(my_runs, REVIEW, wd, audit=40)
# review_blind(REVIEW, annotator="A")
# rescore_pooled(my_runs, REVIEW, ref="llm_no_correction")
# agreement_sample(REVIEW, os.path.join(WORK_DIR, "review_B.csv"), n=60)   # then review_blind(..., "B")
# agreement(REVIEW, os.path.join(WORK_DIR, "review_B.csv"))